# ThaiBev Stock Price Forecasting

This notebook evaluates whether Thai Beverage Public Company Limited (`Y92.SI`) stock prices can be forecast using historical price behavior, technical indicators, and monthly Thai beer-sales data.

The workflow is designed to be reproducible for a GitHub portfolio project: all configurable values are collected in the parameter cell below, raw inputs are loaded from explicit paths, and model results are summarized with MAPE.


## 1. Environment Setup


In [ ]:
%matplotlib inline
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.4f' % x)

import matplotlib.pyplot as plt
import seaborn as sns
import requests
import yfinance as yf

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')


## 2. Project Parameters

Edit this cell to change the stock ticker, date range, train/test split, or local input file.


In [ ]:
PROJECT_ROOT = Path.cwd()
BEER_SALES_FILE = PROJECT_ROOT / 'beer_sales_monthly.csv'
YFINANCE_CACHE_DIR = PROJECT_ROOT / '.yfinance_cache'

TICKER = 'Y92.SI'
START_DATE = '2016-01-01'
END_DATE = '2022-01-01'

DAILY_TEST_DAYS = 280
DAILY_BASELINE_WINDOW = 200
MONTHLY_TEST_MONTHS = 12

EMA_SPAN = 5
RSI_WINDOW = 14
MACD_FAST_SPAN = 12
MACD_SLOW_SPAN = 26
MACD_SIGNAL_SPAN = 9
ARIMA_ORDER = (0, 1, 0)
ARIMA_SEASONAL_ORDER = (1, 0, 1, 8)
EXP_SMOOTHING_SEASONAL_PERIODS = 5

YFINANCE_CACHE_DIR.mkdir(exist_ok=True)
yf.set_tz_cache_location(str(YFINANCE_CACHE_DIR))


## 3. Helper Functions


In [ ]:
def mean_absolute_percentage_error(y_true, y_pred):
    """Return MAPE while ignoring zero actual values."""
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))


def _download_price_from_yahoo_chart_api(ticker, start_date, end_date):
    """Fallback Yahoo Finance chart API download when yfinance returns no rows."""
    period_1 = int(pd.Timestamp(start_date, tz='UTC').timestamp())
    period_2 = int(pd.Timestamp(end_date, tz='UTC').timestamp())
    url = f'https://query1.finance.yahoo.com/v8/finance/chart/{ticker}'
    response = requests.get(
        url,
        params={
            'period1': period_1,
            'period2': period_2,
            'interval': '1d',
            'events': 'history',
            'includeAdjustedClose': 'true',
        },
        headers={'User-Agent': 'Mozilla/5.0'},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    result = payload.get('chart', {}).get('result') or []
    if not result:
        raise ValueError(f'Yahoo chart API returned no price data for {ticker}.')

    result = result[0]
    quote = result['indicators']['quote'][0]
    adjusted_close = result['indicators'].get('adjclose', [{}])[0].get('adjclose')
    prices = pd.DataFrame({
        'Date': pd.to_datetime(result['timestamp'], unit='s').tz_localize('UTC').tz_convert(None).normalize(),
        'Open': quote.get('open'),
        'High': quote.get('high'),
        'Low': quote.get('low'),
        'Close': quote.get('close'),
        'Adj Close': adjusted_close if adjusted_close is not None else quote.get('close'),
        'Volume': quote.get('volume'),
    })
    return prices.dropna(subset=['Close']).reset_index(drop=True)


def get_price(ticker, start_date, end_date):
    """Download OHLCV price data from yfinance, with a direct Yahoo fallback."""
    prices = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        progress=False,
        auto_adjust=False,
    )

    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = [column[0] for column in prices.columns]

    if not prices.empty:
        prices = prices.reset_index()
        prices['Date'] = pd.to_datetime(prices['Date'])
        return prices

    prices = _download_price_from_yahoo_chart_api(ticker, start_date, end_date)
    if prices.empty:
        raise ValueError(f'No price data returned for {ticker}. Check the ticker, dates, or internet connection.')
    return prices


def add_macd(data, column='Adj Close'):
    """Add MACD, signal line, and histogram columns to a copy of data."""
    result = data.copy()
    result['EMA_12'] = result[column].ewm(span=MACD_FAST_SPAN, adjust=False).mean()
    result['EMA_26'] = result[column].ewm(span=MACD_SLOW_SPAN, adjust=False).mean()
    result['MACD'] = result['EMA_12'] - result['EMA_26']
    result['MACD_signal'] = result['MACD'].ewm(span=MACD_SIGNAL_SPAN, adjust=False).mean()
    result['MACD_histogram'] = result['MACD'] - result['MACD_signal']
    return result


def add_rsi(data, column='Adj Close'):
    """Add an RSI column to a copy of data."""
    result = data.copy()
    difference = result[column].diff(1)
    gain = difference.clip(lower=0)
    loss = difference.clip(upper=0)
    average_gain = gain.ewm(com=RSI_WINDOW - 1, min_periods=RSI_WINDOW).mean()
    average_loss = loss.ewm(com=RSI_WINDOW - 1, min_periods=RSI_WINDOW).mean()
    relative_strength = (average_gain / average_loss).abs()
    result['RSI'] = 100 - 100 / (1 + relative_strength)
    return result


def calculate_ema(prices, span, smoothing=2):
    """Calculate an exponential moving average with an initial simple average seed."""
    prices = pd.Series(prices, dtype='float64').reset_index(drop=True)
    if len(prices) < span:
        return [np.nan] * len(prices)

    ema = [np.nan] * (span - 1)
    ema.append(prices.iloc[:span].mean())
    multiplier = smoothing / (1 + span)

    for price in prices.iloc[span:]:
        ema.append((price * multiplier) + ema[-1] * (1 - multiplier))

    return ema


def fit_univariate_linear_regression(train, test, feature, target='Close'):
    """Fit a one-feature linear regression with NumPy and return test predictions."""
    train = train[[feature, target]].dropna().copy()
    test = test[[feature, target]].dropna().copy()

    x_train = train[feature].to_numpy(dtype='float64')
    y_train = train[target].to_numpy(dtype='float64')
    x_mean = x_train.mean()
    x_std = x_train.std()
    if x_std == 0:
        raise ValueError(f'Feature {feature} has zero variance in the training data.')

    x_train_scaled = (x_train - x_mean) / x_std
    slope, intercept = np.polyfit(x_train_scaled, y_train, 1)

    x_test_scaled = (test[feature].to_numpy(dtype='float64') - x_mean) / x_std
    predictions = intercept + slope * x_test_scaled

    return {
        'feature': feature,
        'slope_scaled': slope,
        'intercept': intercept,
        'x_mean': x_mean,
        'x_std': x_std,
        'predictions': predictions,
        'test_index': test.index,
        'test_mape': mean_absolute_percentage_error(test[target], predictions),
    }


## 4. Load Raw Data


In [ ]:
stock_daily = get_price(TICKER, START_DATE, END_DATE)

beer_sales = pd.read_csv(BEER_SALES_FILE)
beer_sales['Date'] = pd.to_datetime(beer_sales['date'], format='%m/%Y')
beer_sales['Month'] = beer_sales['Date'].dt.to_period('M')
beer_sales['sale'] = pd.to_numeric(beer_sales['sale'], errors='raise')
beer_sales = beer_sales[['Month', 'Date', 'sale']].sort_values('Month').reset_index(drop=True)

print(f'Stock rows: {len(stock_daily):,}')
print(f'Beer-sales rows: {len(beer_sales):,}')
stock_daily.head(), beer_sales.head()


## 5. Daily Stock Feature Engineering


In [ ]:
stock_daily['Close_ffill'] = stock_daily['Close'].ffill()
stock_daily['Adj Close'] = stock_daily['Adj Close'].ffill()
stock_daily = add_rsi(stock_daily, column='Adj Close')
stock_daily = add_macd(stock_daily, column='Adj Close')

stock_daily['Close_lag_1'] = stock_daily['Close'].shift(1).ffill()
stock_daily['Close_ma_2'] = stock_daily['Close'].rolling(2).mean()
stock_daily['Close_ema_5'] = calculate_ema(stock_daily['Close'], EMA_SPAN)
stock_daily['Close_ema_5_pandas'] = stock_daily['Close'].ewm(span=EMA_SPAN, adjust=False).mean()

stock_daily.head(10)


## 6. Daily Model Evaluation


In [ ]:
baseline_slice = slice(-DAILY_BASELINE_WINDOW, None)
daily_baseline_results = pd.DataFrame([
    ['EMA5 baseline', mean_absolute_percentage_error(stock_daily['Close'][baseline_slice], stock_daily['Close_ema_5'][baseline_slice])],
    ['MA2 baseline', mean_absolute_percentage_error(stock_daily['Close'][baseline_slice], stock_daily['Close_ma_2'][baseline_slice])],
    ['Lag-1 baseline', mean_absolute_percentage_error(stock_daily['Close'][baseline_slice], stock_daily['Close_lag_1'][baseline_slice])],
], columns=['model', 'mape']).sort_values('mape')

daily_train = stock_daily.iloc[:-DAILY_TEST_DAYS].dropna().copy()
daily_test = stock_daily.iloc[-DAILY_TEST_DAYS:].copy()

daily_lr = fit_univariate_linear_regression(daily_train, daily_test, 'Close_lag_1')
daily_test.loc[daily_lr['test_index'], 'Close_lr'] = daily_lr['predictions']

daily_model_results = pd.DataFrame([
    ['Lagged-close linear regression', daily_lr['test_mape']],
], columns=['model', 'mape'])

daily_baseline_results, daily_model_results


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

arima_model = ARIMA(
    daily_train['Close'],
    order=ARIMA_ORDER,
    trend='t',
    seasonal_order=ARIMA_SEASONAL_ORDER,
    exog=None,
    missing='drop',
).fit()

daily_test['Close_arima'] = arima_model.forecast(steps=len(daily_test)).to_numpy()
daily_arima_mape = mean_absolute_percentage_error(daily_test['Close'], daily_test['Close_arima'])

exp_smoothing_model = ExponentialSmoothing(
    daily_train['Close'],
    damped_trend=True,
    trend='add',
    seasonal='add',
    seasonal_periods=EXP_SMOOTHING_SEASONAL_PERIODS,
    missing='drop',
).fit()

daily_test['Close_exp_smoothing'] = exp_smoothing_model.forecast(steps=len(daily_test)).to_numpy()
daily_exp_smoothing_mape = mean_absolute_percentage_error(daily_test['Close'], daily_test['Close_exp_smoothing'])

daily_model_results = pd.concat([
    daily_model_results,
    pd.DataFrame([
        ['ARIMA', daily_arima_mape],
        ['Exponential smoothing', daily_exp_smoothing_mape],
    ], columns=['model', 'mape'])
], ignore_index=True).sort_values('mape')

daily_model_results


## 7. Monthly Beer-Sales Modeling

The beer-sales data is monthly, so the daily stock data is aggregated to month-end rows before merging. This avoids mixing daily targets with monthly explanatory variables.


In [ ]:
stock_monthly = (
    stock_daily.assign(Month=stock_daily['Date'].dt.to_period('M'))
    .groupby('Month')
    .agg({
        'Date': 'max',
        'Close': 'last',
        'Adj Close': 'last',
        'Volume': 'sum',
        'RSI': 'last',
        'MACD': 'last',
        'MACD_signal': 'last',
        'MACD_histogram': 'last',
    })
    .reset_index()
)

monthly_data = pd.merge(stock_monthly, beer_sales[['Month', 'sale']], on='Month', how='inner')
monthly_data['Close_lag_1'] = monthly_data['Close'].shift(1)
monthly_data['Close_ma_2'] = monthly_data['Close'].rolling(2).mean()
monthly_data['Close_ema_5'] = monthly_data['Close'].ewm(span=EMA_SPAN, adjust=False).mean()
monthly_data = monthly_data.dropna().reset_index(drop=True)

print(f'Monthly modeling rows: {len(monthly_data):,}')
monthly_data.head()


In [ ]:
monthly_train = monthly_data.iloc[:-MONTHLY_TEST_MONTHS].copy()
monthly_test = monthly_data.iloc[-MONTHLY_TEST_MONTHS:].copy()
monthly_predictions = monthly_test[['Month', 'Date', 'Close']].copy()

monthly_results = []
for feature, label in [
    ('sale', 'Beer-sales linear regression'),
    ('MACD', 'MACD linear regression'),
    ('RSI', 'RSI linear regression'),
    ('Close_lag_1', 'Lagged-close linear regression'),
]:
    model = fit_univariate_linear_regression(monthly_train, monthly_test, feature)
    monthly_predictions[f'pred_{feature}'] = model['predictions']
    monthly_results.append([label, model['test_mape']])

monthly_results.append([
    'MA2 baseline',
    mean_absolute_percentage_error(monthly_test['Close'], monthly_test['Close_ma_2']),
])

monthly_exp_smoothing_model = ExponentialSmoothing(
    monthly_train['Close'],
    damped_trend=True,
    trend='add',
    seasonal=None,
    missing='drop',
).fit()

monthly_predictions['pred_exp_smoothing'] = monthly_exp_smoothing_model.forecast(steps=len(monthly_test)).to_numpy()
monthly_results.append([
    'Exponential smoothing',
    mean_absolute_percentage_error(monthly_test['Close'], monthly_predictions['pred_exp_smoothing']),
])

monthly_results = pd.DataFrame(monthly_results, columns=['model', 'mape']).sort_values('mape').reset_index(drop=True)
monthly_results


## 8. Monthly Prediction Plot


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
monthly_train.plot(x='Date', y='Close', ax=ax, label='Train close')
monthly_test.plot(x='Date', y='Close', ax=ax, label='Test close')
monthly_predictions.plot(x='Date', y='pred_MACD', ax=ax, label='MACD prediction')
monthly_predictions.plot(x='Date', y='pred_sale', ax=ax, label='Beer-sales prediction')
ax.set_title('Monthly ThaiBev Close Price Forecasts')
ax.set_xlabel('Date')
ax.set_ylabel('Close price')
ax.legend()
plt.show()


## 9. Summary

The daily workflow benchmarks short-term price-history models. The monthly workflow compares beer sales with technical indicators on the same monthly target. In the latest validation run, the monthly moving-average baseline performed best, while beer sales alone was less predictive than price-history features.
